# Análisis de Secuencias de Interacción
## TFM: Relación entre el uso docente y la valoración del profesorado de una plataforma de aprendizaje en línea

Analiza los flujos de trabajo de los docentes mediante:
1. **Análisis de n-gramas**: patrones frecuentes de transición entre funcionalidades
2. **Análisis comparativo**: diferencias en secuencias entre promotores y detractores
3. **Visualización**: matrices de transición y bigramas distintivos por categoría del NPS

Trabaja con secuencias a nivel de **funcionalidad individual** (`gold_nps_sequences_features`).

**Prerequisito:** ejecutar `00_base_teacher_events.sql` → `02a_gold_table_sequences_features.sql`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from scipy.stats import kruskal
import json
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'font.family': 'serif', 'font.size': 11, 'figure.dpi': 150})
PALETTE = {'detractor': '#d62728', 'passive': '#ff7f0e', 'promoter': '#2ca02c'}
ORDER = ['detractor', 'passive', 'promoter']
LABELS_ES = {'detractor': 'Detractores', 'passive': 'Indiferentes', 'promoter': 'Promotores'}
FIGURES_DIR = os.path.join(os.path.dirname(os.getcwd()), 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)
print(f'Figuras en: {FIGURES_DIR}')


In [ ]:
# ---------------------------------------------------------------------------
# DATOS: acceso requiere infraestructura autorizada (Databricks).
# Los notebooks documentan el codigo analitico con fines de transparencia.
#
# Esquema esperado - gold_nps_sequences_features (una fila por sesion):
#   nps_category           str     'detractor' | 'passive' | 'promoter'
#   session_length         int     Numero de funcionalidades en la sesion
#   feature_sequence       array   Lista ordenada de nombres de funcionalidad
# ---------------------------------------------------------------------------
raise NotImplementedError('Acceso a datos requiere infraestructura autorizada.')


## 1. N-gramas globales (bigramas y trigramas)

In [0]:
def get_ngrams(sequence, n):
    """Extrae n-gramas de una secuencia."""
    return list(zip(*[sequence[i:] for i in range(n)]))

def get_feature_group(name):
    """Asigna grupo funcional a una funcionalidad (espejo del CASE SQL en 00_base_teacher_events.sql).
    Clasificación validada 2026-05-15. Cobertura: 98,7% de eventos clasificados.
    """
    n = name.lower()
    if any(k in n for k in ['speedgrader', 'gradebook', 'grade', 'rubric', 'lmgb']):
        return 'Evaluación y calificación'
    if (n.startswith(('rce', 'page>'))
            or any(k in n for k in ['module', 'syllabus', 'rich content', 'edit page',
                                     'keep editing', 'unpublish', 'drag and drop', 'files',
                                     'pages', 'import content', 'import existing', 'upload',
                                     'cover image', 'add section', 'alt text', 'block editor',
                                     'publish'])):
        return 'Módulos y contenido'
    if any(k in n for k in ['discussion', 'conversation', 'compose']):
        return 'Debates y comunicación'
    if any(k in n for k in ['assignment', 'submission', 'date and time', 'assign to',
                             'due date', 'available from']):
        return 'Tareas y entregas'
    if (n.startswith(('nq |', 'cq |', 'ams'))
            or any(k in n for k in ['quiz', 'question', 'moderation'])):
        return 'Evaluaciones y tests'
    if any(k in n for k in ['navigation', 'dashboard', 'course home page link']):
        return 'Navegación'
    if 'announcement' in n:
        return 'Anuncios'
    if any(k in n for k in ['people', 'roster', 'student', 'attendance', 'group', 'section']):
        return 'Gestión de estudiantes'
    if any(k in n for k in ['analytics', 'insight', 'report']):
        return 'Analítica e informes'
    if (n.startswith(('admin', 'account'))
            or any(k in n for k in ['settings', 'outcomes'])):
        return 'Administración'
    return 'Otros'

# Calcular n-gramas si no fueron pre-computados en una celda anterior.
_already_computed = (
    'bigram_counts' in globals()
    and any(len(bigram_counts.get(c, {})) > 0 for c in ORDER)
)
if not _already_computed:
    bigram_counts  = {cat: Counter() for cat in ORDER}
    trigram_counts = {cat: Counter() for cat in ORDER}
    for _, row in df_seq.iterrows():
        cat = row['nps_category']
        if cat not in ORDER:
            continue
        seq = row['sequence']
        if len(seq) >= 2:
            bigram_counts[cat].update(get_ngrams(seq, 2))
        if len(seq) >= 3:
            trigram_counts[cat].update(get_ngrams(seq, 3))
else:
    print('N-gramas pre-computados — omitiendo loop.')

# --- Bigramas individuales (con grupo funcional entre corchetes) ---
for cat in ORDER:
    print(f'\nTop 10 bigramas — {LABELS_ES[cat].upper()}:')
    for bigram, count in bigram_counts[cat].most_common(10):
        ga = get_feature_group(bigram[0])
        gb = get_feature_group(bigram[1])
        print(f'  {bigram[0]} -> {bigram[1]}: {count:,}  [{ga} -> {gb}]')

# --- Bigramas a nivel de grupo funcional ---
group_bigram_counts = {cat: Counter() for cat in ORDER}
for cat in ORDER:
    for (fa, fb), cnt in bigram_counts[cat].items():
        group_bigram_counts[cat][(get_feature_group(fa), get_feature_group(fb))] += cnt

for cat in ORDER:
    print(f'\nTop 10 bigramas de grupo — {LABELS_ES[cat].upper()}:')
    for bigram, count in group_bigram_counts[cat].most_common(10):
        print(f'  {bigram[0]} -> {bigram[1]}: {count:,}')

# --- Trigramas individuales (con grupos entre corchetes) ---
for cat in ORDER:
    print(f'\nTop 10 trigramas — {LABELS_ES[cat].upper()}:')
    for trigram, count in trigram_counts[cat].most_common(10):
        ga = get_feature_group(trigram[0])
        gb = get_feature_group(trigram[1])
        gc = get_feature_group(trigram[2])
        print(f'  {trigram[0]} -> {trigram[1]} -> {trigram[2]}: {count:,}  [{ga} -> {gb} -> {gc}]')

# --- Trigramas a nivel de grupo funcional ---
group_trigram_counts = {cat: Counter() for cat in ORDER}
for cat in ORDER:
    for (fa, fb, fc), cnt in trigram_counts[cat].items():
        group_trigram_counts[cat][(get_feature_group(fa), get_feature_group(fb), get_feature_group(fc))] += cnt

for cat in ORDER:
    print(f'\nTop 10 trigramas de grupo — {LABELS_ES[cat].upper()}:')
    for trigram, count in group_trigram_counts[cat].most_common(10):
        print(f'  {trigram[0]} -> {trigram[1]} -> {trigram[2]}: {count:,}')

## 2. Matrices de transición entre funcionalidades

In [0]:
def build_transition_matrix(bigram_counter, top_n=15):
    """Construye una matriz de transición normalizada por fila."""
    all_features = Counter()
    for (a, b), cnt in bigram_counter.items():
        all_features[a] += cnt
        all_features[b] += cnt
    top_features = [f for f, _ in all_features.most_common(top_n)]

    matrix = pd.DataFrame(0, index=top_features, columns=top_features, dtype=float)
    for (a, b), cnt in bigram_counter.items():
        if a in top_features and b in top_features:
            matrix.loc[a, b] = cnt

    row_sums = matrix.sum(axis=1).replace(0, 1)
    return matrix.div(row_sums, axis=0)

def build_group_transition_matrix(bigram_counter):
    """Construye una matriz de transición a nivel de grupo funcional."""
    group_counter = Counter()
    for (fa, fb), cnt in bigram_counter.items():
        group_counter[(get_feature_group(fa), get_feature_group(fb))] += cnt
    groups = sorted({g for pair in group_counter for g in pair})
    matrix = pd.DataFrame(0, index=groups, columns=groups, dtype=float)
    for (ga, gb), cnt in group_counter.items():
        matrix.loc[ga, gb] = cnt
    row_sums = matrix.sum(axis=1).replace(0, 1)
    return matrix.div(row_sums, axis=0)

def make_aliases(features):
    """Genera alias cortos F01, F02... para nombres de funcionalidad largos."""
    return {f: f'F{i+1:02d}' for i, f in enumerate(features)}

for cat in ORDER:
    label_es = LABELS_ES[cat]

    # --- Matriz a nivel de funcionalidad individual ---
    tmat = build_transition_matrix(bigram_counts[cat])
    alias = make_aliases(list(tmat.index))
    tmat_plot = tmat.rename(index=alias, columns=alias)

    fig, ax = plt.subplots(figsize=(5, 4))  # Tamaño reducido para docx
    sns.heatmap(tmat_plot, cmap='YlOrRd', ax=ax, vmin=0, vmax=0.5,
                cbar_kws={'label': 'Prob. de transición'}, linewidths=0.3,
                annot=True, fmt='.2f', annot_kws={'size': 7})
    ax.set_title(f'Matriz de transición (funcionalidad individual) — {label_es}', fontsize=10)
    ax.set_xlabel('Funcionalidad destino', fontsize=8)
    ax.set_ylabel('Funcionalidad origen', fontsize=8)
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}06_transition_matrix_{cat}.png', bbox_inches='tight')
    plt.show()

    # --- Leyenda de aliases (con grupo funcional) ---
    print(f'\nLeyenda de aliases — {label_es}:')
    print(f'  {"Alias":<6}  {"Grupo funcional":<35}  Nombre completo')
    print(f'  {"-"*6}  {"-"*35}  {"-"*50}')
    for feat, alias_code in alias.items():
        grupo = get_feature_group(feat)
        print(f'  {alias_code:<6}  {grupo:<35}  {feat}')

    # --- Tabla de datos: matriz individual (con aliases) ---
    print(f'\nTabla de probabilidades de transición (funcionalidad individual) — {label_es}:')
    print(tmat_plot.round(3).to_string())
    tmat_plot.round(3).to_csv(f'{FIGURES_DIR}tabla_transition_individual_{cat}.csv')


    # --- Matriz a nivel de grupo funcional ---
    gmat = build_group_transition_matrix(bigram_counts[cat])

    fig, ax = plt.subplots(figsize=(5, 4))  # Tamaño reducido para docx
    sns.heatmap(gmat, cmap='YlOrRd', ax=ax, vmin=0, vmax=0.5,
                cbar_kws={'label': 'Prob. de transición'}, linewidths=0.5,
                annot=True, fmt='.2f', annot_kws={'size': 7})
    ax.set_title(f'Matriz de transición (grupos funcionales) — {label_es}', fontsize=10)
    ax.set_xlabel('Grupo destino', fontsize=8)
    ax.set_ylabel('Grupo origen', fontsize=8)
    ax.tick_params(axis='x', rotation=45, labelsize=6)
    ax.tick_params(axis='y', rotation=0, labelsize=6)
    ax.set_xticklabels([lbl[:12] + '...' if len(lbl) > 10 else lbl for lbl in gmat.columns], fontsize=6)
    ax.set_yticklabels([lbl[:12] + '...' if len(lbl) > 10 else lbl for lbl in gmat.index], fontsize=6)
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}06_transition_matrix_grupos_{cat}.png', bbox_inches='tight')
    plt.show()

    # --- Tabla de datos: matriz de grupos ---
    print(f'\nTabla de probabilidades de transición (grupos funcionales) — {label_es}:')
    print(gmat.round(3).to_string())
    gmat.round(3).to_csv(f'{FIGURES_DIR}tabla_transition_grupos_{cat}.csv')

## 3. Bigramas distintivos: promotores vs. detractores

In [0]:
def relative_freq(counter):
    total = sum(counter.values())
    return {k: v / total for k, v in counter.items()}

prom_rel = relative_freq(bigram_counts['promoter'])
detr_rel = relative_freq(bigram_counts['detractor'])

all_bigrams = set(prom_rel.keys()) | set(detr_rel.keys())
ratio_df = []
for bg in all_bigrams:
    p = prom_rel.get(bg, 1e-6)
    d = detr_rel.get(bg, 1e-6)
    ga = get_feature_group(bg[0])
    gb = get_feature_group(bg[1])
    ratio_df.append({
        'bigrama': f'{bg[0]} -> {bg[1]}',
        'grupo_origen': ga,
        'grupo_destino': gb,
        'transicion_grupos': f'{ga} -> {gb}',
        'freq_prom': p,
        'freq_detr': d,
        'ratio': p / d,
    })

ratio_df = pd.DataFrame(ratio_df).sort_values('ratio', ascending=False)

print('Top 15 bigramas más característicos de PROMOTORES:')
print(ratio_df.head(15)[['bigrama', 'transicion_grupos', 'freq_prom', 'freq_detr', 'ratio']].to_string(index=False))

print('\nTop 15 bigramas más característicos de DETRACTORES:')
print(ratio_df.tail(15)[['bigrama', 'transicion_grupos', 'freq_prom', 'freq_detr', 'ratio']].to_string(index=False))

In [0]:
top_prom = ratio_df.nlargest(10, 'ratio')
top_detr = ratio_df.nsmallest(10, 'ratio')
combined = pd.concat([top_prom, top_detr]).drop_duplicates().sort_values('ratio').reset_index(drop=True)

# Alias B01, B02... por orden de aparición en el gráfico
combined['alias'] = [f'B{i+1:02d}' for i in range(len(combined))]

fig, ax = plt.subplots(figsize=(6, 5))
colors = [PALETTE['promoter'] if r > 1 else PALETTE['detractor'] for r in combined['ratio']]
ax.barh(range(len(combined)), np.log2(combined['ratio']), color=colors, alpha=0.8)
ax.set_yticks(range(len(combined)))
ax.set_yticklabels(combined['alias'], fontsize=10)
ax.axvline(0, color='gray', linewidth=1, linestyle='--')
ax.set_xlabel('log₂(Ratio Promotores / Detractores)')
ax.set_title('Bigramas distintivos por grupo NPS\n(verde = más frecuente en promotores, rojo = en detractores)')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}07_distinctive_bigrams.png')
plt.show()

# --- Leyenda de aliases ---
print('\nLeyenda de bigramas distintivos:')
print(f'  {"Alias":<6}  {"Transición de grupos":<55}  Bigrama completo')
print(f'  {"-"*6}  {"-"*55}  {"-"*60}')
for _, row in combined.iterrows():
    print(f'  {row["alias"]:<6}  {row["transicion_grupos"]:<55}  {row["bigrama"]}')

## 4. Estadísticas de longitud de sesión por categoría del NPS

In [0]:
df_seq['session_length'] = df_seq['session_length'].astype(float)

session_stats = df_seq.groupby('nps_category')['session_length'].agg(
    ['mean', 'median', 'std', 'count']
).round(2)
session_stats.index = [LABELS_ES.get(i, i) for i in session_stats.index]
print('Longitud de sesión por categoría NPS:')
order_es = [LABELS_ES[c] for c in ORDER]
print(session_stats.loc[[c for c in order_es if c in session_stats.index]])

groups = [df_seq.loc[df_seq.nps_category == cat, 'session_length'] for cat in ORDER]
H, p = kruskal(*groups)
print(f'\nKruskal-Wallis: H={H:.2f}, p={p:.4f}')

fig, ax = plt.subplots(figsize=(7, 5))
data = [df_seq.loc[df_seq.nps_category == cat, 'session_length'].clip(upper=20) for cat in ORDER]
bp = ax.boxplot(data, labels=[LABELS_ES[c] for c in ORDER], patch_artist=True)
for patch, cat in zip(bp['boxes'], ORDER):
    patch.set_facecolor(PALETTE[cat])
    patch.set_alpha(0.7)
ax.set_ylabel('Número de funcionalidades por sesión')
ax.set_title('Longitud de sesión por categoría NPS (cap. en 20)')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}08_session_length_by_nps.png')
plt.show()

## 5. Exportar resultados

In [0]:
ratio_df.to_csv(f'{FIGURES_DIR}tabla_bigrams_ratio.csv', index=False)
session_stats.to_csv(f'{FIGURES_DIR}tabla_session_stats.csv')
print('Análisis de secuencias completado. Resultados en:', FIGURES_DIR)